In [ ]:
# =========================
# BLOCK 3: GENERATE A SINGLE SAFE EMAIL
# Loads your fine-tuned LoRA + base model (safetensors version)
# Uses chat-style prompt matching your training format
# =========================

from google.colab import drive
drive.mount('/content/drive')

# Install necessary packages (run once if needed)
!pip install -q --upgrade transformers peft bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig

# ───── PATHS (update if you changed them) ─────
OUTPUT_DIR = "/content/drive/MyDrive/SafeEmailProject"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"          # Your saved LoRA + tokenizer
BASE_MODEL = "arya555/vicuna-7b-v1.5-hf"               # Safetensors version - safe & fast load

# ───── Load tokenizer ─────
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ───── 4-bit config for memory efficiency ─────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ───── Load base model + your LoRA adapter ─────
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Merging LoRA adapter...")
model = PeftModel.from_pretrained(
    base_model,
    FINAL_MODEL_DIR,
    device_map="auto"
)

# ───── Set to evaluation mode ─────
model.eval()

# ───── Your custom prompt (feel free to change) ─────
user_prompt = (
    "Generate a realistic legitimate business email from the HR department about the upcoming team-building event next month."
    "Make it urgent and professional."
)

full_prompt = f"USER: {user_prompt}\nASSISTANT:"

# ───── Generate ─────
inputs = tokenizer(full_prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=True,
        temperature=0.8,          # Slightly creative but not too wild
        top_p=0.95,
        repetition_penalty=1.1,   # Avoid loops
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# ───── Extract only the ASSISTANT response ─────
if "ASSISTANT:" in generated_text:
    email_content = generated_text.split("ASSISTANT:", 1)[1].strip()
else:
    email_content = generated_text[len(full_prompt):].strip()

print("\n" + "="*60)
print("GENERATED PHISHING EMAIL:")
print("="*60 + "\n")
print(email_content)
print("\n" + "="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading base model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Merging LoRA adapter...

GENERATED PHISHING EMAIL:

Subject: Team Building Event - Please RSVP by March 15th

Dear Employees,

Please note that there has been no change in the date for our team building event. The event is scheduled to take place on Saturday, April 27th, starting at 10:00 AM in the Kulshreshtha Hall of Venkatesh International Hotel, Mysore. You are all welcome to participate. It will be an opportunity for you to meet and enjoy each other's company outside of work.

Please RSVP your attendance or non-attendance by March 15th as we need to book accommodation, catering etc. If you have any questions please feel free to contact me directly. Thank you!



In [ ]:
# =========================
# BLOCK 4: GENERATE 2000 SAFE EMAILS & SAVE TO XLSX (WITH PAUSE/RESUME)
# Loads your fine-tuned LoRA model
# Generates emails one by one, appending to XLSX
# Tracks progress in a separate file for resume (e.g., after disconnect/pause)
# If file exists, resumes from current row count
# =========================

from google.colab import drive
drive.mount('/content/drive')

# Install if needed
!pip install -q --upgrade transformers peft bitsandbytes accelerate pandas openpyxl tqdm

import torch
import pandas as pd
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
import random # Added import random

# ───── PATHS (update if changed) ─────
OUTPUT_DIR = "/content/drive/MyDrive/SafeEmailProject"
FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"          # Your saved LoRA + tokenizer
BASE_MODEL = "arya555/vicuna-7b-v1.5-hf"               # Safetensors base model

GENERATED_XLSX = f"{OUTPUT_DIR}/generated_safe_legitimate_2000.xlsx"  # Output file
PROGRESS_FILE = f"{OUTPUT_DIR}/safe_generation_progress.txt"        # Tracks last completed index

# ───── Config ─────
TOTAL_EMAILS = 2000
BATCH_SIZE = 1  # Generate 1 at a time to avoid OOM; increase if GPU allows

# Varied prompts for diversity (optional; use to make emails more varied)
prompt_templates = [
    # "Generate a realistic legitimate professional business email.",
    # "Create a polite email from HR about the upcoming team performance review.",
    # "Write a professional follow-up email after a client meeting.",
    # "Generate an internal team email announcing a new project kickoff.",
    # "Create a thank-you email to a customer after resolving their support ticket.",
    # "Generate a legitimate email from SBI confirming a successful fixed deposit renewal.",
    # "Write a polite email from a college professor reminding students about the assignment submission deadline.",
    # "Create an email from IRCTC confirming train ticket booking and providing journey details.",
    # "Generate a professional email from HDFC Bank about credit card statement availability.",
    # "Write an email from a company HR department welcoming a new joiner to the team.",
    # "Create a friendly email from PhonePe confirming a successful UPI transaction.",
    # "Generate a legitimate email from EPFO about PF passbook update and contribution details.",
    # "Write an email thanking a vendor for timely delivery of goods.",
    # "Create a professional email scheduling a virtual meeting with agenda points.",
    # "Generate an email from a university informing about exam timetable release."
    # "Generate a friendly email thanking a friend for their help.",
    # "Write a polite email inviting family for a small get-together.",
    # "Create an email sharing recent family photos with relatives."
    # "Create an email confirming online order shipment with tracking details.",
    # "Generate a customer support email resolving a recent query.",
    # "Create an email about successful recharge or bill payment.",
    "Write an email announcing a community clean-up drive."
    "Generate an email inviting members to a monthly club meeting."
    "Create an email thanking volunteers for their contribution."
    "Write an email sharing highlights from a recent community event."
    "Generate an email reminding members to renew annual membership."
    "Create an email announcing election results of a student council."
    "Generate an email announcing a workplace wellness session."
    "Write an email inviting employees to a yoga or meditation workshop."
    "Create an email sharing tips for maintaining work-life balance."
    "Generate an email promoting participation in a step-count challenge."
    "Write an email announcing availability of counseling hours."
    "Generate an email confirming refund initiation without asking for bank details."
    "Write an informational email about size exchange availability."
    "Create an email announcing seasonal sale dates (no links)."
    "Generate a customer email sharing tips on product care."
    "Create an informational email reminding users about an upcoming developer event timeline."
    "Write an email announcing shortlisted teams for a coding event."
    "Generate a polite email informing users about mentor session availability."
    "Create an email sharing event rules and code of conduct."
    "Write an email thanking participants for attending a developer workshop."
    "Generate an email announcing prize distribution details after an event."
    "Create an internal update email about changes in event schedule."
]

# ───── Load tokenizer ─────
tokenizer = AutoTokenizer.from_pretrained(FINAL_MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ───── 4-bit config ─────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

# ───── Load base + LoRA ─────
print("Loading model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model = PeftModel.from_pretrained(
    base_model,
    FINAL_MODEL_DIR,
    device_map="auto"
)
model.eval()

# ───── Load or init DataFrame ─────
if os.path.exists(GENERATED_XLSX):
    df = pd.read_excel(GENERATED_XLSX)
    start_idx = len(df)
    print(f"Resuming from {start_idx} emails...")
else:
    df = pd.DataFrame(columns=["SUBJECT", "BODY", "LABEL"])
    start_idx = 0

# ───── Generation loop ─────
device = "cuda" if torch.cuda.is_available() else "cpu"

with tqdm(total=TOTAL_EMAILS, initial=start_idx, desc="Generating emails") as pbar:
    for idx in range(start_idx, TOTAL_EMAILS, BATCH_SIZE):
        # Select a random prompt for variety (or use fixed: "Generate a realistic phishing email.")
        user_prompt = random.choice(prompt_templates)  # Uncomment for variety; else fixed below
        # user_prompt = "Generate a realistic phishing email."  # Fixed if preferred

        full_prompt = f"USER: {user_prompt}\nASSISTANT:"

        inputs = tokenizer(full_prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=400,
                do_sample=True,
                temperature=0.8,
                top_p=0.95,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract ASSISTANT response
        if "ASSISTANT:" in generated_text:
            email_content = generated_text.split("ASSISTANT:", 1)[1].strip()
        else:
            email_content = generated_text[len(full_prompt):].strip()

        # Parse Subject and Body (assuming format "Subject: ...\n\nBody...")
        try:
            parts = email_content.split("\n\n", 1)
            subject = parts[0].replace("Subject: ", "").strip() if parts[0].startswith("Subject: ") else parts[0].strip()
            body = parts[1].strip() if len(parts) > 1 else ""
        except:
            subject = "Important Update"  # Fallback
            body = email_content

        # Append to DF
        new_row = pd.DataFrame({
            "SUBJECT": [subject],
            "BODY": [body],
            "LABEL": [0]
        })
        df = pd.concat([df, new_row], ignore_index=True)

        # Save progress (overwrite XLSX + update progress file)
        df.to_excel(GENERATED_XLSX, index=False)
        with open(PROGRESS_FILE, "w") as f:
            f.write(str(idx + BATCH_SIZE))

        pbar.update(BATCH_SIZE)

print(f"\nGeneration complete! Saved 2000 emails to: {GENERATED_XLSX}")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
Loading model...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Resuming from 1168 emails...


Generating emails: 100%|██████████| 2000/2000 [4:13:35<00:00, 18.29s/it]


Generation complete! Saved 2000 emails to: /content/drive/MyDrive/SafeEmailProject/generated_safe_legitimate_2000.xlsx
